# Iris Classifier - MLOps Training Notebook

Este notebooook (ahora si funca o q) entrena un clasificador sobre el dataset Iris usando scikit-learn.
Está diseñado para ser ejecutado automáticamenteeeeee por la plataforma MLOps cuando
se detectan cambios vía webhook de GitHub.

**Flujo automático:**
1. Push al repositorio → GitHub Webhook
2. La plataforma descarga este notebook
3. Lo ejecuta con Papermill inyectando parámetros
4. Registra el modelo en MLflow
5. Si accuracy >= 70%, despliega automáticamente

In [ ]:
# ── Configuración del modelo ──────────────────────────────────────────
# Estos valores identifican el modelo en la plataforma.
# Cambia VERSION cada vez que quieras desplegar una nueva versión.

MODEL_NAME = "iris-classifier"
VERSION = "1"

In [ ]:
# ── Parámetros inyectados por Papermill ──────────────────────────────
# La plataforma sobreescribe estos valores automáticamente al ejecutar.
# Los valores por defecto permiten ejecutar el notebook manualmente.

MODEL_OUTPUT_PATH = "model.joblib"
PIPELINE_ID = "local-run"
MLFLOW_TRACKING_URI = "http://mlflow:5000"
MLFLOW_RUN_ID = ""

In [ ]:
# ── Preprocesamiento ─────────────────────────────────────────────────
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Cargar dataset
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"Dataset: {X.shape[0]} muestras, {X.shape[1]} features")
print(f"Features: {feature_names}")
print(f"Clases: {list(target_names)}")
print(f"Distribución: {dict(zip(*np.unique(y, return_counts=True)))}")

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)
print(f"\nTrain: {X_train.shape[0]} muestras")
print(f"Test:  {X_test.shape[0]} muestras")

# Escalar features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("\nScaling aplicado.")

In [ ]:
# ── Entrenamiento ────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Entrenar modelo
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train_scaled, y_train)

# Evaluar
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))
print(f"Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# ── Exportar modelo y registrar métricas ─────────────────────────────
import joblib
import mlflow
from sklearn.pipeline import Pipeline

# Crear pipeline completo (scaler + modelo) para que el model-server
# pueda hacer predicciones sin preprocesar manualmente.
full_pipeline = Pipeline([
    ("scaler", scaler),
    ("classifier", model),
])

# Guardar modelo en la ruta que la plataforma espera
joblib.dump(full_pipeline, MODEL_OUTPUT_PATH)
print(f"Modelo guardado en: {MODEL_OUTPUT_PATH}")

# Registrar métricas en MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

if MLFLOW_RUN_ID:
    with mlflow.start_run(run_id=MLFLOW_RUN_ID):
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("n_estimators", 100)
        mlflow.log_metric("max_depth", 5)
        mlflow.log_metric("train_size", len(X_train))
        mlflow.log_metric("test_size", len(X_test))
        mlflow.log_param("model_type", "RandomForestClassifier")
        mlflow.log_param("model_name", MODEL_NAME)
        mlflow.log_param("version", VERSION)
    print(f"Métricas registradas en MLflow (run_id={MLFLOW_RUN_ID})")
else:
    print("MLFLOW_RUN_ID no proporcionado. Ejecución local, skip MLflow.")

print(f"\n{'='*50}")
print(f"  Modelo: {MODEL_NAME} v{VERSION}")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Pipeline ID: {PIPELINE_ID}")
print(f"{'='*50}")